# 01 -- Zero-Shot Classification: DeBERTa-v3-large (Test Split)
## Model: MoritzLaurer/deberta-v3-large-zeroshot-v1.1-all-33

Zero-Shot NLI-basierte Klassifikation auf dem **frozen Test-Split**.

Dieses Notebook verwendet deutsche NLI-Verbalisierungen als Candidate Labels und
einen Confidence-Threshold für die "Andere"-Kategorie.

Kein Training nötig — das Modell wird direkt für Inferenz verwendet.

**Voraussetzung:** GPU-Runtime aktiviert, `HF_TOKEN` in Colab Secrets hinterlegt.

In [ ]:
# === SETUP ===
import os, sys
from pathlib import Path

# Repo klonen / aktualisieren
REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

# Dependencies
!pip install -q transformers datasets huggingface_hub scikit-learn matplotlib seaborn tqdm pandas

# Google Drive mounten (persistente Reports)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# pipeline_utils importierbar machen (liegt in euroBERT_210m/)
PIPELINE_DIR = f"{REPO}/Python/classification_pipeline/euroBERT_210m"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

# --- Output-Pfade auf zero_shot_bert umleiten ---
_DRIVE_ZS = Path("/content/drive/MyDrive/thesis_reports/zero_shot_bert")
_LOCAL_ZS = Path(f"{REPO}/Python/classification_pipeline/zero_shot_bert/local_data")

pu.BASE_DIR = _DRIVE_ZS if pu.DRIVE_AVAILABLE else _LOCAL_ZS
pu.REPORTS_DIR = pu.BASE_DIR / "reports"
pu.OPTUNA_DB_DIR = pu.BASE_DIR / "optuna_db"
pu.CLASSIFICATION_OUTPUT_DIR = pu.BASE_DIR / "classification_output"
pu.VISUALIZATIONS_DIR = pu.BASE_DIR / "visualizations"

pu.ensure_drive_dirs()

# HuggingFace Login
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

print(f"Reports-Ordner: {pu.REPORTS_DIR}")
print("Setup abgeschlossen.")

In [ ]:
# ===== MODEL CONFIG =====
MODEL_ID = "MoritzLaurer/deberta-v3-large-zeroshot-v1.1-all-33"
MODEL_SHORT_NAME = "deberta_v3_large"
MODEL_TYPE = "zero-shot"

# Dieses Notebook evaluiert auf dem FROZEN Test-Split
EVALUATE_ON = "test"

# Batch-Größe für Klassifikation (kleiner = weniger VRAM, langsamer)
BATCH_SIZE = 16

# ===== NLI CONFIG =====
HYPOTHESIS_TEMPLATE = "Dieser Text handelt {}."

# Candidate Labels: None = aus label_mapping in Zelle 4 ableiten
CANDIDATE_LABELS = None

# Confidence-Schwellenwert: max NLI-Score < Threshold -> "Andere"
CONFIDENCE_THRESHOLD = 0.4

# ===== MODEL INFO (für Report) =====
MODEL_INFO = {
    "huggingface_id": MODEL_ID,
    "language": "English",
    "max_tokens": 512,
    "parameters": "304M",
    "notes": "DeBERTa-v3-large, trained on 33 NLI datasets. Supports FP16. English-only.",
}

In [ ]:
# ===== LABEL VERBALISIERUNG =====
# Links: Original-Label im Datensatz
# Rechts: NLI-Verbalisierung (wird in die Hypothese eingesetzt)
#
# Hypothese: "Dieser Text handelt {VERBALISIERUNG}."
#
# "Andere" wird NICHT verbalisiert — stattdessen per Confidence-Threshold zugewiesen.

LABEL_MAPPING = {
    "Klima / Energie":          "vom Klima, Klimawandel oder der Energieversorgung",
    "Zuwanderung":              "von Zuwanderung oder Migration",
    "Renten":                   "von der Rente oder dem Rentensystem",
    "Soziales Gefälle":         "vom Sozialen Gefälle oder von sozialer Ungleicheit",
    "AfD/Rechte":               "von der Brandmauer, der AfD oder dem Rechtsextremismus",
    "Arbeitslosigkeit":         "von Arbeitslosigkeit",
    "Wirtschaftslage":          "von der Wirtschaftslage oder der Zukunft der Deutschen Wirtschaft",
    "Politikverdruss":          "von Politikverdruss, dem Vertrauen in die Demokraite oder der Interesse für Politik bei den Bürgern",
    "Gesundheitswesen, Pflege": "vom Gesundheitswesen, der Pflege oder Krankenversicherungen",
    "Kosten/Löhne/Preise":      "von steigenden Preisen und Lebenshaltungskosten oder von Löhnen",
    "Ukraine/Krieg/Russland":   "vom Ukraine Krieg oder von Russland",
    "Bundeswehr/Verteidigung":  "von der Bundeswehr, der Verteidigung Deutschlands oder Investitionen in die Rüstung",
}

# Vorschau der Hypothesen
print("Vorschau der NLI-Hypothesen:")
print("-" * 60)
for orig, verb in LABEL_MAPPING.items():
    print(f"  {orig:30s} \u2192 \"{HYPOTHESIS_TEMPLATE.format(verb)}\"")
print(f"\n  {'Andere':30s} \u2192 per Threshold (< {CONFIDENCE_THRESHOLD})")

In [ ]:
# Daten laden
data = pu.load_data(
    split_mode="percentage",
    eval_fraction=0.2,
    random_seed=42,
    load_raw=False,
    label_mapping=LABEL_MAPPING,
)

eval_df = data[EVALUATE_ON]

# Candidate Labels: nur die 12 konkreten (ohne "Andere")
if CANDIDATE_LABELS is None:
    CANDIDATE_LABELS = list(data["label_mapping"].values())

# Alle Labels inkl. "Andere" für Evaluation
ALL_LABELS = CANDIDATE_LABELS + ["Andere"]

print(f"\nEvaluiere auf '{EVALUATE_ON}' Split: {len(eval_df)} Artikel")
print(f"NLI Candidate Labels ({len(CANDIDATE_LABELS)}): {CANDIDATE_LABELS}")
print(f"Evaluation Labels ({len(ALL_LABELS)}): inkl. 'Andere' per Threshold")

In [ ]:
# Modell laden
import torch
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1

classifier = hf_pipeline(
    "zero-shot-classification",
    model=MODEL_ID,
    device=device,
    torch_dtype=torch.float16 if device == 0 else torch.float32,
)

# Architektur-Details aus config.json extrahieren
model_config = pu.extract_model_config(classifier)

print(f"Modell geladen: {MODEL_ID}")
print(f"Device: {'GPU' if device == 0 else 'CPU'}")
print(f"Tokenizer max length: {classifier.tokenizer.model_max_length}")
print(f"Architecture: {model_config}")

In [ ]:
# Klassifikation (Volltext)
from tqdm.auto import tqdm

def classify_batch(texts, batch_size=BATCH_SIZE):
    """Zero-Shot Klassifikation mit Confidence-Threshold für 'Andere'."""
    predictions = [None] * len(texts)
    threshold_count = 0
    non_empty_indices = [i for i, t in enumerate(texts) if t.strip()]
    non_empty_texts = [texts[i] for i in non_empty_indices]

    for start in tqdm(range(0, len(non_empty_texts), batch_size), desc="Classifying"):
        batch_texts = non_empty_texts[start:start + batch_size]
        batch_indices = non_empty_indices[start:start + batch_size]

        results = classifier(
            batch_texts,
            candidate_labels=CANDIDATE_LABELS,
            hypothesis_template=HYPOTHESIS_TEMPLATE,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]

        for idx, r in zip(batch_indices, results):
            top_label = r["labels"][0]
            top_score = r["scores"][0]
            if top_score < CONFIDENCE_THRESHOLD:
                predictions[idx] = "Andere"
                threshold_count += 1
            else:
                predictions[idx] = top_label

    empty_count = sum(1 for p in predictions if p is None)
    if empty_count > 0:
        print(f"  {empty_count} leere Texte -> 'Andere'")
    print(f"  {threshold_count} Artikel per Threshold (< {CONFIDENCE_THRESHOLD}) -> 'Andere'")
    return [p if p is not None else "Andere" for p in predictions]


texts = eval_df["text"].fillna("").tolist()
true_labels = eval_df["label"].tolist()

timer = pu.ExperimentTimer()
with timer:
    predictions = classify_batch(texts)

print(f"\nKlassifikation abgeschlossen: {timer.duration_formatted}")
print(f"Durchsatz: {timer.articles_per_second(len(texts)):.2f} Artikel/Sekunde")

In [ ]:
# Evaluation
metrics = pu.evaluate(
    true_labels,
    predictions,
    labels=ALL_LABELS,
    experiment_name=EVALUATE_ON,
)

pu.print_metrics(metrics, f"Zero-Shot DeBERTa-v3-large \u2014 {EVALUATE_ON} Split")

In [ ]:
# Confusion Matrix
pu.plot_confusion_matrix(
    metrics,
    title=f"Zero-Shot DeBERTa-v3-large ({EVALUATE_ON})",
)

In [ ]:
# Report generieren
report_path = pu.generate_report(
    model_name=f"{MODEL_SHORT_NAME}_zeroshot",
    model_type=MODEL_TYPE,
    metrics=metrics,
    timer=timer,
    model_info=MODEL_INFO,
    candidate_labels=ALL_LABELS,
    hypothesis_template=HYPOTHESIS_TEMPLATE,
    split_config=data["split_config"],
    label_mapping=data["label_mapping"],
    model_config=model_config,
    experiment_notes=(
        "Zero-Shot NLI-Klassifikation mit DeBERTa-v3-large auf Volltext. "
        "Texte werden automatisch auf 512 Tokens gek\u00fcrzt (inverted pyramid). "
        f"'Andere' per Confidence-Threshold ({CONFIDENCE_THRESHOLD}) zugewiesen."
    ),
)

print(f"\nReport gespeichert: {report_path}")

In [ ]:
# Summary
print("=" * 70)
print(f"  Model:           {MODEL_ID}")
print(f"  Type:            {MODEL_TYPE}")
print(f"  Split:           {EVALUATE_ON} ({len(eval_df)} Artikel)")
print(f"  F1 Macro:        {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:     {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  Dauer:           {timer.duration_formatted}")
print(f"  Artikel/Sek:     {timer.articles_per_second(len(eval_df)):.2f}")
print("=" * 70)